# 4) One Line Quiz Agent (No Answer Key)

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Run quick one-question quizzes by topic and log answers without revealing correctness.

## Simple rules

- Ask one question.
- Log answer.
- Let user choose topic.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "04_quiz_memory.json"

QUESTION_BANK = {
    "python": [
        "Difference between list and tuple (1 line)?",
        "What does len(some_dict) return?",
        "What does range(3) produce conceptually?",
    ],
    "dsa": [
        "Define time complexity in one line.",
        "What is an invariant (1 line)?",
        "What does FIFO mean?",
    ],
    "db": [
        "What does a primary key guarantee?",
        "What is a foreign key used for?",
        "What is normalization (1 line)?",
    ],
}

def decide_next_action(topic_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(topic_text):
        return Action("terminate", {})

    topic = topic_text.strip().lower()
    if topic not in QUESTION_BANK:
        return Action("notify", {"message": f"Pick topic: {', '.join(QUESTION_BANK.keys())}"})

    q = random.choice(QUESTION_BANK[topic])
    memory["topic"] = topic
    memory["last_q"] = q
    return Action("notify", {"message": f"Topic: {topic}\nQuestion: {q}\n(Answer is logged; no checking.)"})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    attempts = memory.get("attempts", [])

    Tools.notify("One Line Quiz Agent is running.")
    Tools.notify("Choose topic: python / dsa / db. Type 'stop' to end.")

    topic_text = Tools.ask("Topic?")
    while True:
        action = decide_next_action(topic_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            memory["attempts"] = attempts
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved quiz log. Bye!")
            break

        ans = Tools.ask("Your answer (one line):")
        attempts.append({"topic": memory["topic"], "question": memory["last_q"], "answer": ans, "ts": time.time()})
        Tools.notify("Logged.")
        topic_text = Tools.ask("Next topic? (or stop)")

run_agent()


## Easy extensions

- Add confidence rating.
- Export to CSV.
- Add more topics.